# Valve Integrity Analytics — Notebook 2: Analysis Engine

**Project:** Well Integrity Surveillance Analytics Suite  
**Input:** `valve_tests_clean.csv` (produced by Notebook 1)  
**Purpose:** Turn raw test records into actionable well-level intelligence.

---
### What this notebook covers
1. Load the clean dataset
2. **Module A — Barrier Status Engine:** classify each valve's current health (OK / At Risk / Critical)
3. **Module B — Overdue Test Detector:** flag wells whose last test exceeded the recommended interval
4. **Module C — Historical Trend Analyser:** track each well's failure pattern over time
5. **Module D — Master Integrity Summary Table:** one row per well-valve with all flags combined + risk score
6. Save outputs for Notebook 3 (visualisations) and Notebook 4 (reporting)

---
### Domain reminder
- Each well has 4 valves: **SCSSV, PWV, PMV, AMV**
- A valve is a **barrier** — if it leaks (Failed), well integrity is compromised
- Tests must happen every **180 days** (PWV/PMV/AMV) or **210 days** (SCSSV)
- An overdue test = unknown current barrier status → treated as a risk

## Step 0 — Imports & settings

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', '{:.1f}'.format)

# Reference date — used for overdue calculations
# Set to today, or pin to a specific date for reproducibility
REFERENCE_DATE = pd.Timestamp('2026-03-16')

print(f'Analysis reference date: {REFERENCE_DATE.strftime("%d %b %Y")}')
print('Libraries ready.')

Analysis reference date: 16 Mar 2026
Libraries ready.


## Step 1 — Load clean dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

CLEAN_PATH = '/content/drive/MyDrive/valve_integrity_project/data/valve_tests_clean.csv'

df = pd.read_csv(CLEAN_PATH, parse_dates=['test_date'])

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range: {df["test_date"].min().date()} → {df["test_date"].max().date()}')
df.head(5)

Mounted at /content/drive
Loaded: 780 rows × 13 columns
Date range: 2023-10-31 → 2026-03-11


,well_id,asset,valve_type,test_date,pressure_before_bled,pressure_after_bled,pressure_after_buildup,test_pressure,leak_rate,delta_p,result,recommended_interval_days,remarks
0,BP-01,British Palm,AMV,2024-01-16,921,348,350,2499,8,2,Passed,180,No leak observed
1,BP-01,British Palm,AMV,2024-07-23,809,275,288,2530,22,13,Passed,180,"Minor leak, within limit"
2,BP-01,British Palm,AMV,2025-01-11,985,233,238,2794,11,5,Passed,180,No abnormality detected
3,BP-01,British Palm,AMV,2025-08-10,772,433,438,2700,16,5,Passed,180,Test satisfactory
4,BP-01,British Palm,AMV,2026-03-08,1094,411,420,1986,22,9,Passed,180,Test satisfactory


---
## Module A — Barrier Status Engine

**Goal:** For every well-valve combination, look at the **most recent test only** and classify its barrier status.

| Last Result | Barrier Status | Meaning |
|---|---|---|
| Passed, ΔP well within limit | `HEALTHY` | Valve holding, no concern |
| Passed, ΔP close to limit (>75%) | `MONITOR` | Passed but trending toward limit |
| Failed | `AT RISK` | Valve leaked on last test |
| Failed 2+ times in history | `CRITICAL` | Repeat failure — priority intervention |

In [3]:
# ── A1: Get the most recent test per well-valve ──
latest_tests = (
    df.sort_values('test_date')
      .groupby(['well_id', 'asset', 'valve_type'])
      .last()
      .reset_index()
)

print(f'Well-valve combinations: {len(latest_tests)}')
print(f'Expected: 39 wells × 4 valves = 156')
latest_tests[['well_id', 'asset', 'valve_type', 'test_date', 'delta_p', 'leak_rate', 'result']].head(8)

Well-valve combinations: 156
Expected: 39 wells × 4 valves = 156


,well_id,asset,valve_type,test_date,delta_p,leak_rate,result
0,BP-01,British Palm,AMV,2026-03-08,9,22,Passed
1,BP-01,British Palm,PMV,2026-03-03,8,17,Passed
2,BP-01,British Palm,PWV,2026-02-26,20,24,Passed
3,BP-01,British Palm,SCSSV,2026-02-24,5,8,Passed
4,BP-02,British Palm,AMV,2026-03-08,9,17,Passed
5,BP-02,British Palm,PMV,2026-02-28,12,14,Passed
6,BP-02,British Palm,PWV,2026-03-02,11,16,Passed
7,BP-02,British Palm,SCSSV,2026-03-08,10,13,Passed


In [4]:
# ── A2: Count total failures per well-valve across ALL history ──
failure_history = (
    df[df['result'] == 'Failed']
      .groupby(['well_id', 'valve_type'])
      .size()
      .reset_index(name='total_failures_ever')
)

# Merge failure history onto latest tests
latest_tests = latest_tests.merge(
    failure_history, on=['well_id', 'valve_type'], how='left'
)
latest_tests['total_failures_ever'] = latest_tests['total_failures_ever'].fillna(0).astype(int)

print('Failure history merged.')
print(latest_tests[latest_tests['total_failures_ever'] > 0][
    ['well_id', 'valve_type', 'result', 'total_failures_ever']
].to_string(index=False))

Failure history merged.
well_id valve_type result  total_failures_ever
  BP-02      SCSSV Passed                    1
  BP-05        PWV Passed                    1
  BP-07        AMV Passed                    1
  BP-10        PMV Passed                    1
  BP-10      SCSSV Passed                    1
  BP-13        PWV Failed                    1
  EY-03      SCSSV Passed                    1
  EY-06        PWV Passed                    1
  EY-08        AMV Passed                    2
 FIS-03        AMV Failed                    1
 FIS-07      SCSSV Passed                    1
 FIS-11        PWV Passed                    1
 FIS-14        AMV Passed                    1
 FIS-16        PWV Passed                    1
 FIS-16      SCSSV Passed                    1


In [5]:
# ── A3: Classify barrier status ──

def classify_barrier_status(row):
    """
    Rules (evaluated in priority order):
    1. CRITICAL  — last test Failed AND has failed 2+ times ever
    2. AT RISK   — last test Failed
    3. MONITOR   — last test Passed BUT delta_p > 75% of leak_rate (trending toward limit)
    4. HEALTHY   — last test Passed and delta_p comfortably within limit
    """
    if row['result'] == 'Failed' and row['total_failures_ever'] >= 2:
        return 'CRITICAL'
    elif row['result'] == 'Failed':
        return 'AT RISK'
    elif row['delta_p'] > 0.75 * row['leak_rate']:
        return 'MONITOR'
    else:
        return 'HEALTHY'

latest_tests['barrier_status'] = latest_tests.apply(classify_barrier_status, axis=1)

print('=== BARRIER STATUS DISTRIBUTION ===')
print(latest_tests['barrier_status'].value_counts().to_string())

print('\n=== AT RISK & CRITICAL VALVES ===')
at_risk = latest_tests[latest_tests['barrier_status'].isin(['AT RISK', 'CRITICAL'])]
display(at_risk[['well_id', 'asset', 'valve_type', 'test_date',
                  'delta_p', 'leak_rate', 'result',
                  'total_failures_ever', 'barrier_status']])

=== BARRIER STATUS DISTRIBUTION ===
barrier_status
HEALTHY    117
MONITOR     37
AT RISK      2

=== AT RISK & CRITICAL VALVES ===


,well_id,asset,valve_type,test_date,delta_p,leak_rate,result,total_failures_ever,barrier_status
50,BP-13,British Palm,PWV,2026-03-03,58,20,Failed,1,AT RISK
96,FIS-03,France Intial State,AMV,2026-02-25,46,16,Failed,1,AT RISK


---
## Module B — Overdue Test Detector

**Goal:** For each well-valve, calculate how many days since the last test and flag if it exceeds the recommended interval.

- `days_since_last_test` = Reference Date − Last Test Date
- `days_overdue` = days_since_last_test − recommended_interval_days (negative = still within schedule)
- `overdue_flag` = True if days_since_last_test > recommended_interval_days

In [6]:
# ── B1: Calculate days since last test ──
latest_tests['days_since_last_test'] = (
    REFERENCE_DATE - latest_tests['test_date']
).dt.days

latest_tests['days_overdue'] = (
    latest_tests['days_since_last_test'] - latest_tests['recommended_interval_days']
)

latest_tests['overdue_flag'] = latest_tests['days_overdue'] > 0

# Summary
overdue_count = latest_tests['overdue_flag'].sum()
total = len(latest_tests)
print(f'Overdue valve tests : {overdue_count} / {total} ({overdue_count/total*100:.1f}%)')
print(f'Within schedule     : {total - overdue_count} / {total}')

Overdue valve tests : 0 / 156 (0.0%)
Within schedule     : 156 / 156


In [7]:
# ── B2: Overdue wells detail ──
overdue_wells = (
    latest_tests[latest_tests['overdue_flag'] == True]
    .sort_values('days_overdue', ascending=False)
    [['well_id', 'asset', 'valve_type', 'test_date',
      'recommended_interval_days', 'days_since_last_test', 'days_overdue', 'result']]
)

print(f'=== OVERDUE TESTS ({len(overdue_wells)} records) ===')
display(overdue_wells)

=== OVERDUE TESTS (0 records) ===


,well_id,asset,valve_type,test_date,recommended_interval_days,days_since_last_test,days_overdue,result


In [8]:
# ── B3: Overdue breakdown by asset ──
print('=== OVERDUE TESTS BY ASSET ===')
overdue_by_asset = latest_tests.groupby('asset').apply(
    lambda x: pd.Series({
        'Total Valves'  : len(x),
        'Overdue'       : x['overdue_flag'].sum(),
        'Overdue %'     : round(x['overdue_flag'].mean() * 100, 1),
        'Max Days Overdue': x.loc[x['overdue_flag'], 'days_overdue'].max() if x['overdue_flag'].any() else 0
    })
).reset_index()
display(overdue_by_asset)

print('\n=== OVERDUE TESTS BY VALVE TYPE ===')
overdue_by_valve = latest_tests.groupby('valve_type').apply(
    lambda x: pd.Series({
        'Total'   : len(x),
        'Overdue' : x['overdue_flag'].sum(),
        'Overdue %': round(x['overdue_flag'].mean() * 100, 1)
    })
).reset_index()
display(overdue_by_valve)

=== OVERDUE TESTS BY ASSET ===


,asset,Total Valves,Overdue,Overdue %,Max Days Overdue
0,British Palm,56.0,0.0,0.0,0.0
1,European Yard,32.0,0.0,0.0,0.0
2,France Intial State,68.0,0.0,0.0,0.0



=== OVERDUE TESTS BY VALVE TYPE ===


,valve_type,Total,Overdue,Overdue %
0,AMV,39.0,0.0,0.0
1,PMV,39.0,0.0,0.0
2,PWV,39.0,0.0,0.0
3,SCSSV,39.0,0.0,0.0


---
## Module C — Historical Trend Analyser

**Goal:** Look beyond the latest test — understand each well-valve's behaviour across all 5 test cycles.

Key metrics computed:
- `total_tests` — how many tests recorded
- `total_failures` — how many times it failed
- `failure_rate_pct` — failures as % of total tests
- `avg_delta_p` — average pressure build-up over time (trending up = deteriorating)
- `delta_p_trend` — is ΔP increasing over time? (linear slope)
- `consecutive_failures` — how many failures in a row (most recent)

In [9]:
# ── C1: Per well-valve aggregate stats ──

def compute_trend_slope(series):
    """Fit a linear slope to a series of values. Positive = rising trend."""
    if len(series) < 2:
        return 0.0
    x = np.arange(len(series))
    slope = np.polyfit(x, series, 1)[0]
    return round(slope, 3)

def count_consecutive_failures(result_series):
    """Count trailing consecutive failures (most recent first)."""
    count = 0
    for r in reversed(result_series.tolist()):
        if r == 'Failed':
            count += 1
        else:
            break
    return count

trend_stats = df.sort_values('test_date').groupby(['well_id', 'asset', 'valve_type']).apply(
    lambda g: pd.Series({
        'total_tests'             : len(g),
        'total_failures'          : (g['result'] == 'Failed').sum(),
        'failure_rate_pct'        : round((g['result'] == 'Failed').mean() * 100, 1),
        'avg_delta_p'             : round(g['delta_p'].mean(), 1),
        'max_delta_p'             : g['delta_p'].max(),
        'delta_p_slope'           : compute_trend_slope(g['delta_p']),
        'consecutive_failures'    : count_consecutive_failures(g['result']),
        'first_test_date'         : g['test_date'].min(),
        'last_test_date'          : g['test_date'].max(),
    })
).reset_index()

# Flag rising trend: slope > 0.5 PSI per test cycle
trend_stats['rising_dp_trend'] = trend_stats['delta_p_slope'] > 0.5

print(f'Trend stats computed for {len(trend_stats)} well-valve combinations.')
print('\n=== WELLS WITH RISING DELTA-P TREND (deteriorating) ===')
rising = trend_stats[trend_stats['rising_dp_trend']].sort_values('delta_p_slope', ascending=False)
display(rising[['well_id', 'valve_type', 'avg_delta_p', 'max_delta_p', 'delta_p_slope', 'total_failures']].head(15))

Trend stats computed for 156 well-valve combinations.

=== WELLS WITH RISING DELTA-P TREND (deteriorating) ===


,well_id,valve_type,avg_delta_p,max_delta_p,delta_p_slope,total_failures
50,BP-13,PWV,20.6,58,7.8,1
96,FIS-03,AMV,19.8,46,5.5,1
67,EY-03,SCSSV,13.4,44,4.1,1
22,BP-06,PWV,12.8,21,3.6,0
23,BP-06,SCSSV,5.6,15,3.5,0
70,EY-04,PWV,6.0,19,3.3,0
151,FIS-16,SCSSV,17.2,50,3.3,1
130,FIS-11,PWV,14.8,46,3.2,1
54,BP-14,PWV,9.8,19,3.1,0
18,BP-05,PWV,14.0,49,3.1,1


In [10]:
# ── C2: Spotlight — top 5 worst well-valve combinations by failure rate ──
print('=== TOP 10 WORST WELL-VALVE COMBINATIONS (by failure rate) ===')
worst = (
    trend_stats[trend_stats['total_failures'] > 0]
    .sort_values(['total_failures', 'failure_rate_pct'], ascending=False)
    .head(10)
)
display(worst[['well_id', 'asset', 'valve_type', 'total_tests',
               'total_failures', 'failure_rate_pct', 'avg_delta_p', 'delta_p_slope']])

=== TOP 10 WORST WELL-VALVE COMBINATIONS (by failure rate) ===


,well_id,asset,valve_type,total_tests,total_failures,failure_rate_pct,avg_delta_p,delta_p_slope
84,EY-08,European Yard,AMV,5,2,40.0,19.8,3.0
7,BP-02,British Palm,SCSSV,5,1,20.0,18.2,1.8
18,BP-05,British Palm,PWV,5,1,20.0,14.0,3.1
24,BP-07,British Palm,AMV,5,1,20.0,11.0,-2.2
37,BP-10,British Palm,PMV,5,1,20.0,14.4,-0.0
39,BP-10,British Palm,SCSSV,5,1,20.0,12.0,3.0
50,BP-13,British Palm,PWV,5,1,20.0,20.6,7.8
67,EY-03,European Yard,SCSSV,5,1,20.0,13.4,4.1
78,EY-06,European Yard,PWV,5,1,20.0,17.6,2.4
96,FIS-03,France Intial State,AMV,5,1,20.0,19.8,5.5


---
## Module D — Master Integrity Summary Table

**Goal:** Combine all three modules into a single, engineer-ready table.

One row = one well-valve combination. Contains:
- Latest test result & barrier status (from Module A)
- Overdue flags & days overdue (from Module B)
- Historical trend metrics (from Module C)
- **Risk Score (0–100)** — composite score for prioritising interventions

### Risk Score logic
| Factor | Weight | Points |
|---|---|---|
| Last test Failed | 40 | 40 |
| Repeat failure (2+ ever) | 20 | 20 |
| Overdue test | 20 | 20 |
| Rising ΔP trend | 10 | 10 |
| ΔP > 50% of leak rate | 10 | 10 |
| **Max possible** | | **100** |

In [11]:
# ── D1: Merge all modules ──
summary = latest_tests.merge(
    trend_stats[[
        'well_id', 'valve_type',
        'total_tests', 'total_failures', 'failure_rate_pct',
        'avg_delta_p', 'max_delta_p', 'delta_p_slope',
        'consecutive_failures', 'rising_dp_trend'
    ]],
    on=['well_id', 'valve_type'],
    how='left'
)

print(f'Master summary shape: {summary.shape}')

Master summary shape: (156, 26)


In [12]:
# ── D2: Compute Risk Score ──

def compute_risk_score(row):
    score = 0
    # Last test failed → +40
    if row['result'] == 'Failed':
        score += 40
    # Repeat failure in history → +20
    if row['total_failures_ever'] >= 2:
        score += 20
    # Test overdue → +20
    if row['overdue_flag']:
        score += 20
    # Rising delta_p trend → +10
    if row['rising_dp_trend']:
        score += 10
    # Delta P > 50% of leak rate on latest test → +10
    if row['leak_rate'] > 0 and row['delta_p'] > 0.5 * row['leak_rate']:
        score += 10
    return score

summary['risk_score'] = summary.apply(compute_risk_score, axis=1)

# ── D3: Risk Level label from score ──
def risk_level(score):
    if score >= 60:
        return 'CRITICAL'
    elif score >= 40:
        return 'HIGH'
    elif score >= 20:
        return 'MEDIUM'
    else:
        return 'LOW'

summary['risk_level'] = summary['risk_score'].apply(risk_level)

print('=== RISK LEVEL DISTRIBUTION ===')
print(summary['risk_level'].value_counts().to_string())

print('\n=== RISK SCORE STATS ===')
print(summary['risk_score'].describe().to_string())

=== RISK LEVEL DISTRIBUTION ===
risk_level
LOW         103
MEDIUM       51
CRITICAL      2

=== RISK SCORE STATS ===
count   156.0
mean     10.4
std      10.1
min       0.0
25%       0.0
50%      10.0
75%      20.0
max      60.0


In [13]:
# ── D4: Final column selection & ordering ──
final_cols = [
    'well_id', 'asset', 'valve_type',
    'test_date', 'result', 'delta_p', 'leak_rate',
    'barrier_status',
    'days_since_last_test', 'recommended_interval_days', 'days_overdue', 'overdue_flag',
    'total_tests', 'total_failures', 'total_failures_ever',
    'failure_rate_pct', 'avg_delta_p', 'max_delta_p',
    'delta_p_slope', 'rising_dp_trend', 'consecutive_failures',
    'risk_score', 'risk_level'
]

master_summary = summary[final_cols].sort_values(
    ['risk_score', 'days_overdue'], ascending=False
).reset_index(drop=True)

print('=== MASTER INTEGRITY SUMMARY — TOP 20 HIGHEST RISK ===')
display(master_summary.head(20))

=== MASTER INTEGRITY SUMMARY — TOP 20 HIGHEST RISK ===


,well_id,asset,valve_type,test_date,result,delta_p,leak_rate,barrier_status,days_since_last_test,recommended_interval_days,...,total_failures,total_failures_ever,failure_rate_pct,avg_delta_p,max_delta_p,delta_p_slope,rising_dp_trend,consecutive_failures,risk_score,risk_level
0,FIS-03,France Intial State,AMV,2026-02-25,Failed,46,16,AT RISK,19,180,...,1,1,20.0,19.8,46,5.5,True,1,60,CRITICAL
1,BP-13,British Palm,PWV,2026-03-03,Failed,58,20,AT RISK,13,180,...,1,1,20.0,20.6,58,7.8,True,1,60,CRITICAL
2,EY-08,European Yard,AMV,2026-03-01,Passed,2,8,HEALTHY,15,180,...,2,2,40.0,19.8,59,3.0,True,0,30,MEDIUM
3,FIS-01,France Intial State,PWV,2026-02-25,Passed,10,19,HEALTHY,19,180,...,0,0,0.0,9.6,15,0.8,True,0,20,MEDIUM
4,FIS-04,France Intial State,AMV,2026-02-25,Passed,13,16,MONITOR,19,180,...,0,0,0.0,7.6,16,1.3,True,0,20,MEDIUM
5,FIS-05,France Intial State,AMV,2026-02-25,Passed,11,14,MONITOR,19,180,...,0,0,0.0,8.6,16,0.7,True,0,20,MEDIUM
6,BP-01,British Palm,PWV,2026-02-26,Passed,20,24,MONITOR,18,180,...,0,0,0.0,9.8,20,1.7,True,0,20,MEDIUM
7,EY-06,European Yard,PWV,2026-02-26,Passed,12,12,MONITOR,18,180,...,1,1,20.0,17.6,59,2.4,True,0,20,MEDIUM
8,BP-09,British Palm,PMV,2026-02-27,Passed,17,22,MONITOR,17,180,...,0,0,0.0,9.2,17,3.1,True,0,20,MEDIUM
9,EY-06,European Yard,PMV,2026-02-27,Passed,11,12,MONITOR,17,180,...,0,0,0.0,11.6,17,1.0,True,0,20,MEDIUM


In [14]:
# ── D5: Well-level rollup (worst valve per well drives the well's risk) ──
well_summary = (
    master_summary
    .sort_values('risk_score', ascending=False)
    .groupby(['well_id', 'asset'])
    .agg(
        worst_barrier_status = ('barrier_status', 'first'),
        valves_at_risk        = ('result', lambda x: (x == 'Failed').sum()),
        valves_overdue        = ('overdue_flag', 'sum'),
        max_risk_score        = ('risk_score', 'max'),
        well_risk_level       = ('risk_level', 'first'),
        worst_valve           = ('valve_type', 'first'),
    )
    .reset_index()
    .sort_values('max_risk_score', ascending=False)
)

print('=== WELL-LEVEL RISK RANKING (top 15) ===')
display(well_summary.head(15))

=== WELL-LEVEL RISK RANKING (top 15) ===


,well_id,asset,worst_barrier_status,valves_at_risk,valves_overdue,max_risk_score,well_risk_level,worst_valve
12,BP-13,British Palm,AT RISK,1,0,60,CRITICAL,PWV
24,FIS-03,France Intial State,AT RISK,1,0,60,CRITICAL,AMV
21,EY-08,European Yard,HEALTHY,0,0,30,MEDIUM,AMV
5,BP-06,British Palm,MONITOR,0,0,20,MEDIUM,PMV
0,BP-01,British Palm,MONITOR,0,0,20,MEDIUM,PWV
1,BP-02,British Palm,MONITOR,0,0,20,MEDIUM,PMV
2,BP-03,British Palm,HEALTHY,0,0,20,MEDIUM,AMV
36,FIS-15,France Intial State,HEALTHY,0,0,20,MEDIUM,SCSSV
37,FIS-16,France Intial State,HEALTHY,0,0,20,MEDIUM,PMV
25,FIS-04,France Intial State,MONITOR,0,0,20,MEDIUM,AMV


---
## Step 6 — Save all outputs

In [18]:
OUTPUT_DIR = '/content/drive/MyDrive/valve_integrity_project/outputs/'

# 1. Master integrity summary (valve-level)
master_path = OUTPUT_DIR + 'integrity_summary_valve_level.csv'
master_summary.to_csv(master_path, index=False)
print(f'✅ Valve-level summary saved  → {master_path}')

# 2. Well-level rollup
well_path = OUTPUT_DIR + 'integrity_summary_well_level.csv'
well_summary.to_csv(well_path, index=False)
print(f'✅ Well-level summary saved   → {well_path}')

# 3. Exception report — only AT RISK, CRITICAL, or overdue
exception_report = master_summary[
    (master_summary['barrier_status'].isin(['AT RISK', 'CRITICAL'])) |
    (master_summary['overdue_flag'] == True)
].copy()
exception_path = OUTPUT_DIR + 'exception_report.csv'
exception_report.to_csv(exception_path, index=False)
print(f'✅ Exception report saved     → {exception_path}')
print(f'   ({len(exception_report)} wells flagged for engineer review)')

# 4. Full enriched dataset (all 780 rows + trend stats joined)
df_enriched = df.merge(
    trend_stats[['well_id', 'valve_type', 'total_failures',
                 'failure_rate_pct', 'avg_delta_p', 'delta_p_slope', 'rising_dp_trend']],
    on=['well_id', 'valve_type'], how='left'
)
enriched_path = OUTPUT_DIR + 'valve_tests_enriched.csv'
df_enriched.to_csv(enriched_path, index=False)
print(f'✅ Enriched full dataset saved → {enriched_path}')

✅ Valve-level summary saved  → /content/drive/MyDrive/valve_integrity_project/outputs/integrity_summary_valve_level.csv
✅ Well-level summary saved   → /content/drive/MyDrive/valve_integrity_project/outputs/integrity_summary_well_level.csv
✅ Exception report saved     → /content/drive/MyDrive/valve_integrity_project/outputs/exception_report.csv
   (2 wells flagged for engineer review)
✅ Enriched full dataset saved → /content/drive/MyDrive/valve_integrity_project/outputs/valve_tests_enriched.csv


## Notebook 2 Summary

| Output File | Contents | Used in |
|---|---|---|
| `integrity_summary_valve_level.csv` | 156 rows, one per well-valve, all risk flags | NB3 charts, NB4 PDF |
| `integrity_summary_well_level.csv` | 39 rows, one per well, worst-valve drives risk | NB3 charts |
| `exception_report.csv` | Only failed / overdue wells | NB4 report |
| `valve_tests_enriched.csv` | All 780 rows + trend stats | NB3 timeline charts |

---
**Risk Score recap:**
- **0–19** → LOW — routine monitoring
- **20–39** → MEDIUM — schedule review
- **40–59** → HIGH — priority test or inspection
- **60–100** → CRITICAL — immediate engineer action

**Next → Notebook 3:** Visualisations — charts and heatmaps for all analysis outputs.